# Qorvix — CUDA backend test on a real GPU

Builds Qorvix with the real CUDA backend (`nvcc`) and runs its GPU self-tests on the actual
device. Use this to verify CUDA **execution** — the dev box and CI only compile-check it.

## ⚠️ Use a GPU runtime, NOT a TPU

CUDA runs only on NVIDIA GPUs. Set **Runtime → Change runtime type → Hardware accelerator = T4
GPU** (or any GPU). A **TPU** runtime has no `nvcc` / no CUDA device and the first cell will stop
with an error.

Run the cells in order. Each step is separate so any failure is easy to spot.

## 1. Check for an NVIDIA GPU

In [1]:
import subprocess, sys
ok = subprocess.run(['bash','-lc','nvidia-smi >/dev/null 2>&1']).returncode == 0
if not ok:
    print('ERROR: no NVIDIA GPU detected.')
    print('Colab: Runtime > Change runtime type > Hardware accelerator = T4 GPU (NOT TPU), then rerun.')
    raise SystemExit(1)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
!nvcc --version | grep -i release || echo 'WARNING: nvcc not found (GPU but no CUDA toolkit)'

Tesla T4, 580.82.07, 15360 MiB
Cuda compilation tools, release 12.8, V12.8.93


## 2. Install build tools (ninja, gcc-12, recent CMake)

In [2]:
!apt-get -qq update >/dev/null 2>&1
!apt-get -qq install -y ninja-build g++-12 gcc-12 >/dev/null 2>&1
!pip -q install --upgrade cmake >/dev/null 2>&1
!cmake --version | head -1

cmake version 4.4.0


## 3. Clone the repository

In [3]:
!rm -rf /content/qorvix
!git clone --depth 1 https://github.com/Boominathan2355/QorVix.git /content/qorvix
!git -C /content/qorvix log --oneline -1

Cloning into '/content/qorvix'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 156 (delta 0), reused 128 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 130.07 KiB | 1.13 MiB/s, done.
62af150 (grafted, HEAD -> main, origin/main, origin/HEAD) Add Colab notebook for the CUDA GPU test


## 4. Configure with CUDA enabled

No vcpkg needed: the CUDA executable and its libraries use only `nvcc` + a C++23 host compiler
(Boost/Catch2 are just for the HTTP API and unit tests, which are turned off here).

In [4]:
!cd /content/qorvix && CC=gcc-12 CXX=g++-12 cmake -S . -B build-cuda -G Ninja \
  -DCMAKE_BUILD_TYPE=Release \
  -DQORVIX_ENABLE_CUDA=ON \
  -DQORVIX_BUILD_TESTS=OFF \
  -DCMAKE_CUDA_HOST_COMPILER=g++-12

-- The CXX compiler identification is GNU 12.3.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/g++-12 - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Looking for a CUDA compiler
-- Looking for a CUDA compiler - /usr/local/cuda/bin/nvcc
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include (found version "12.8.93")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- qorvix_cuda: building the real CUDA backend (nvcc).
-- qorvix_runtime: OpenMP enabled
-- Configuring done (10.0s)
-- Genera

## 5. Build

Watch for `qorvix_cuda: building the real CUDA backend (nvcc).` in the configure output above, and
for `Building CUDA object ... cuda_backend.cu.o` here.

In [5]:
!cmake --build /content/qorvix/build-cuda -j$(nproc)

[16/37] Building CUDA object cuda/CMakeFiles/qorvix_cuda.dir/src/cuda_backend.cu.oKp.o
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
[37/37] Linking CXX executable core/qorvixs/qorvix.dir/cmake_device_link.o/example_plugin.cpp.o


## 6. Run the GPU self-tests on the real device

`qorvix gpu` enumerates the device and runs a custom scale kernel (host→device→kernel→host) and a
cuBLAS SGEMM, checking results on the host. Exit code is nonzero if any self-test fails.

In [6]:
!/content/qorvix/build-cuda/core/qorvix gpu

CUDA support: built in.
Devices: 1

  [0] Tesla T4
      compute capability : 7.5
      SMs                : 40
      memory (free/total): 14 GB / 15 GB

Self-test (scale kernel): PASS — scale kernel host<->device round-trip verified over 1024 floats
Self-test (cuBLAS GEMM):  PASS — cuBLAS SGEMM (A=I) verified on a 4x4 matrix


## What this proves (and what it doesn't)

PASS on both self-tests means the CUDA **backend** — device/memory management, a custom kernel
launch, and cuBLAS — executes correctly on this GPU.

It is **not** end-to-end GPU inference yet: the transformer forward pass still runs on CPU. Moving
that validated forward pass onto the GPU is **Phase 8**, and this notebook becomes the way to
benchmark it. Copy the output back to the chat for interpretation.